## setup

In [1]:
%pip install vaderSentiment
%pip install openpyxl

# import stuff
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### patient data

In [2]:
# load kaggle patient data
patient = pd.read_excel('data/patient_feedback_dataset.xlsx')

print('Shape:', patient.shape)
patient.head()

Shape: (1000, 5)


,Theme,Feedback,Sentiment,Satisfaction,Readmission
0,discharge,Discharge instructions were very clear,1,4,1
1,communication,I felt the doctor used too much medical jargon,1,5,0
2,medication,I appreciated the reminders about my medicines,1,4,0
3,discharge,Instructions after discharge were not helpful,0,1,1
4,discharge,Instructions after discharge were not helpful,0,2,1


In [3]:
patient.describe()
# sentiment is binary
# satisfaction scale 1-5

,Sentiment,Satisfaction,Readmission
count,1000.00000,1000.000000,1000.000000
mean,0.58400,3.418000,0.288000
std,0.49314,1.493153,0.453058
min,0.00000,1.000000,0.000000
25%,0.00000,2.000000,0.000000
50%,1.00000,4.000000,0.000000
75%,1.00000,5.000000,1.000000
max,1.00000,5.000000,1.000000


In [4]:
patient['Sentiment'].value_counts()

Sentiment
1    584
0    416
Name: count, dtype: int64

In [5]:
patient['Satisfaction'].value_counts()

Satisfaction
5    326
4    258
1    174
2    144
3     98
Name: count, dtype: int64

In [6]:
patient.groupby('Theme')[['Sentiment', 'Satisfaction']].mean()

,Sentiment,Satisfaction
Theme,,
communication,0.779026,3.977528
discharge,0.563025,3.373950
medication,0.606695,3.451883
wait_time,0.378906,2.843750


In [7]:
patient['Feedback'].unique()
# only 20 unique feedback sentences

array(['Discharge instructions were very clear',
       'I felt the doctor used too much medical jargon',
       'I appreciated the reminders about my medicines',
       'Instructions after discharge were not helpful',
       'Waiting time was too long', 'The pharmacy service was excellent',
       'I was seen by the doctor quickly',
       'I had to wait for hours before being admitted',
       'Doctor explained my condition clearly',
       'I forgot my schedule due to unclear medication notes',
       'Nurses were very patient and communicative',
       'Discharge paperwork was easy to follow',
       'The staff rushed me during discharge',
       'The staff made me feel comfortable',
       'Medication instructions were confusing',
       'I felt confused about my discharge process',
       'Waiting for tests took forever',
       'The nurse explained my prescriptions well',
       'Appointments started on time',
       "I did not understand the doctor's instructions"], dtype=objec

In [8]:
analyzer = SentimentIntensityAnalyzer()

# Work on a copy and exclude missing/empty feedback
patient_scored = patient.copy()
patient_scored["text"] = (
    patient_scored["Feedback"].astype("string").str.strip()
)
patient_scored = patient_scored.loc[
    patient_scored["text"].notna()
    & patient_scored["text"].ne("")
].copy()

# Score the review text only
patient_scored["compound"] = patient_scored["text"].apply(
    lambda text: analyzer.polarity_scores(text)["compound"]
)

def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

patient_scored["predicted_sentiment"] = (
    patient_scored["compound"].apply(sentiment_label)
)

patient_scored[
    ["Theme", "text", "Sentiment", "compound", "predicted_sentiment"]
].head(10)

,Theme,text,Sentiment,compound,predicted_sentiment
0,discharge,Discharge instructions were very clear,1,0.4391,positive
1,communication,I felt the doctor used too much medical jargon,1,0.0000,neutral
2,medication,I appreciated the reminders about my medicines,1,0.5106,positive
3,discharge,Instructions after discharge were not helpful,0,-0.3252,negative
4,discharge,Instructions after discharge were not helpful,0,-0.3252,negative
5,wait_time,Waiting time was too long,0,0.0000,neutral
6,discharge,Discharge instructions were very clear,1,0.4391,positive
7,medication,I appreciated the reminders about my medicines,1,0.5106,positive
8,discharge,Instructions after discharge were not helpful,0,-0.3252,negative
9,medication,The pharmacy service was excellent,1,0.5719,positive


In [21]:
unique_feedback = patient_scored.drop_duplicates("text").copy()

# Provisional mapping: confirm what 0 and 1 mean in the dataset
unique_feedback["dataset_label"] = (
    unique_feedback["Sentiment"].map({0: "negative", 1: "positive"})
)

display(unique_feedback[
    ["text", "dataset_label", "compound", "predicted_sentiment"]
])

# These are disagreements to investigate, not proven model errors
disagreements = unique_feedback[
    unique_feedback["dataset_label"]
    != unique_feedback["predicted_sentiment"]
]

display(disagreements[
    ["text", "dataset_label", "compound", "predicted_sentiment"]
])

,text,dataset_label,compound,predicted_sentiment
0,Discharge instructions were very clear,positive,0.4391,positive
1,I felt the doctor used too much medical jargon,positive,0.0000,neutral
2,I appreciated the reminders about my medicines,positive,0.5106,positive
3,Instructions after discharge were not helpful,negative,-0.3252,negative
5,Waiting time was too long,negative,0.0000,neutral
9,The pharmacy service was excellent,positive,0.5719,positive
10,I was seen by the doctor quickly,positive,0.0000,neutral
11,I had to wait for hours before being admitted,negative,0.1027,positive
12,Doctor explained my condition clearly,positive,0.4019,positive
13,I forgot my schedule due to unclear medication...,negative,-0.2500,negative


,text,dataset_label,compound,predicted_sentiment
1,I felt the doctor used too much medical jargon,positive,0.0000,neutral
5,Waiting time was too long,negative,0.0000,neutral
10,I was seen by the doctor quickly,positive,0.0000,neutral
11,I had to wait for hours before being admitted,negative,0.1027,positive
14,Nurses were very patient and communicative,positive,0.0000,neutral
18,The staff rushed me during discharge,negative,0.0000,neutral
27,I felt confused about my discharge process,positive,-0.3182,negative
30,Waiting for tests took forever,negative,0.0000,neutral
37,Appointments started on time,positive,0.0000,neutral
72,I did not understand the doctor's instructions,negative,0.0000,neutral


### yelp data

In [ ]:
yelp = pd.read_csv("data/Yelp Restaurant Reviews.csv")

print('Shape:', yelp.shape)
yelp.head()

Shape: (19896, 4)


,Yelp URL,Rating,Date,Review Text
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,1/22/2022,All I can say is they have very good ice cream...
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,6/26/2022,Nice little local place for ice cream.My favor...
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,8/7/2021,A delicious treat on a hot day! Staff was very...
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,7/28/2016,This was great service and a fun crew! I got t...
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,6/23/2015,This is one of my favorite places to get ice c...


In [ ]:
# vader thing
analyzer = SentimentIntensityAnalyzer()

# find polarity score of each review & add to a new df
scores = yelp['Review Text'].apply(lambda x: analyzer.polarity_scores(str(x))).apply(pd.Series)

# check sentiment scores
scores.head()

,neg,neu,pos,compound
0,0.000,0.663,0.337,0.8803
1,0.045,0.625,0.330,0.9531
2,0.000,0.624,0.376,0.9224
3,0.019,0.776,0.206,0.9400
4,0.013,0.741,0.246,0.9940


In [ ]:
# add sentiment scores to og df
yelp = pd.concat([yelp, scores], axis = 1)

# check yelp df after concat sentiment sccores
yelp.head()

,Yelp URL,Rating,Date,Review Text,neg,neu,pos,compound
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,1/22/2022,All I can say is they have very good ice cream...,0.000,0.663,0.337,0.8803
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,6/26/2022,Nice little local place for ice cream.My favor...,0.045,0.625,0.330,0.9531
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,8/7/2021,A delicious treat on a hot day! Staff was very...,0.000,0.624,0.376,0.9224
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,7/28/2016,This was great service and a fun crew! I got t...,0.019,0.776,0.206,0.9400
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,6/23/2015,This is one of my favorite places to get ice c...,0.013,0.741,0.246,0.9940
